# Building an AI Explorer App with Streamlit

## A Step-by-Step Guide

**FAMNIT AI Workshop -- Day 2**

---

In this notebook, you will build an interactive web application from scratch using **Streamlit**. By the end, you will have a fully working app that lets users explore:

1. **Embeddings** -- How text becomes numbers that capture meaning
2. **Chunking** -- How to split documents for processing
3. **Vector Search** -- How to search by meaning using ChromaDB

### What is Streamlit?

Streamlit is a Python library that turns ordinary Python scripts into interactive web applications. There is **no HTML, CSS, or JavaScript** required -- everything is pure Python.

### Why Streamlit?

- **Pure Python** -- If you can write a Python script, you can build a web app
- **Live reload** -- The app updates automatically when you change the code
- **Built for data science** -- First-class support for charts, tables, and ML models
- **Free deployment** -- Deploy to Streamlit Cloud directly from GitHub

### How this notebook works

Streamlit apps run as **separate processes**, not inside Jupyter. So our workflow is:

1. Write code into `app.py` using the `%%writefile` magic command
2. Run the app with `streamlit run app.py`
3. Open your browser to see the result
4. Come back here, add more code, and repeat

Each step builds on the previous one. By the final step, you will have the complete app.

---

## Step 0: Setup

First, let's install all the libraries we need. This may take a minute or two.

In [ ]:
!pip install streamlit langchain langchain-community langchain-huggingface \
    chromadb sentence-transformers scikit-learn plotly pandas numpy -q

### How `%%writefile` works

The `%%writefile app.py` magic command at the top of a cell tells Jupyter: **"Don't run this code -- write it to a file instead."**

```python
%%writefile app.py
import streamlit as st
st.title("Hello!")
```

This creates (or overwrites) `app.py` with the cell contents. Then we run the app from a separate cell:

```python
!streamlit run app.py --server.port 8501
```

Open your browser at **http://localhost:8501** to see the app.

> **Tip:** When you update `app.py`, Streamlit detects the change automatically. Just click **"Rerun"** in the browser (or press **R**) to see the new version. You do NOT need to restart the server each time.

---

## Step 1: Your First Streamlit App (Hello World)

Let's start with the simplest possible app to see how Streamlit works.

In [ ]:
%%writefile app.py
import streamlit as st

st.title("My AI Workshop App")
st.write("Welcome! This is my first Streamlit app.")

# Text input -- creates an interactive text box
name = st.text_input("What's your name?")

if name:
    st.write(f"Hello, {name}! Let's explore AI together.")

### Run the app now!

Run the cell below to start the Streamlit server. Then open **http://localhost:8501** in your browser.

In [ ]:
# Start the Streamlit server in the background
# On Windows, use start. On Mac/Linux, use &
import subprocess, sys, os

# Kill any existing Streamlit process first
if sys.platform == "win32":
    os.system("taskkill /f /im streamlit.exe 2>nul")
else:
    os.system("pkill -f 'streamlit run' 2>/dev/null")

# Start Streamlit
proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print(f"Streamlit started (PID: {proc.pid})")
print("Open your browser at: http://localhost:8501")

### What just happened?

With just 5 lines of Python, you created an interactive web application. Here is what each function does:

| Function | What it does |
|----------|-------------|
| `st.title()` | Displays a large heading |
| `st.write()` | Displays text, data, charts -- almost anything |
| `st.text_input()` | Creates a text box and returns what the user typed |

Streamlit re-runs your entire script from top to bottom every time the user interacts with a widget. This is different from Jupyter -- there is no cell-by-cell execution.

---

## Step 2: Adding a Sidebar and Navigation

Real apps have multiple pages. Streamlit makes this easy with a sidebar and simple `if/elif` logic.

Run the cell below to update `app.py`. Then go to your browser and click **"Rerun"** (or press **R**).

In [ ]:
%%writefile app.py
import streamlit as st

# Configure the page -- must be the FIRST Streamlit command
st.set_page_config(page_title="AI Explorer", page_icon="\U0001f9e0", layout="wide")

# Sidebar navigation
st.sidebar.title("AI Explorer")
page = st.sidebar.radio(
    "Navigate",
    ["Home", "Embeddings", "Chunking", "Vector Search"]
)

# Page routing -- simple if/elif
if page == "Home":
    st.title("AI Explorer App")
    st.markdown("""
    ### What we'll explore:
    1. **Embeddings** -- How text becomes numbers
    2. **Chunking** -- How to split documents
    3. **Vector Search** -- How to search by meaning
    
    Use the sidebar on the left to navigate between pages.
    """)

elif page == "Embeddings":
    st.title("Embeddings")
    st.write("Coming soon...")

elif page == "Chunking":
    st.title("Chunking Strategies")
    st.write("Coming soon...")

elif page == "Vector Search":
    st.title("Vector Search")
    st.write("Coming soon...")

### Go check your browser!

You should now see a sidebar on the left. Click each page name to switch between them.

### What's new here?

| Function | What it does |
|----------|-------------|
| `st.set_page_config()` | Sets the browser tab title, icon, and layout. Must be the first Streamlit call. |
| `st.sidebar.title()` | Adds a title inside the sidebar |
| `st.sidebar.radio()` | Creates a radio button group in the sidebar and returns the selected option |
| `st.markdown()` | Renders Markdown text (bold, lists, headers, etc.) |

The navigation pattern is simple: `st.sidebar.radio()` returns a string, and we use `if/elif` to show the right content. This is how most Streamlit apps handle multiple pages.

---

## Step 3: Building the Embeddings Page

Now let's build a real feature. The Embeddings page will let users type two texts and see how similar they are, using a real embedding model.

**New concepts in this step:**
- `@st.cache_resource` -- Load the model once and reuse it across reruns
- `st.button()` -- A clickable button
- `st.metric()` -- Display a number with a label
- `st.success()`, `st.info()`, `st.warning()` -- Colored alert boxes
- `st.expander()` -- A collapsible section

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np

st.set_page_config(page_title="AI Explorer", page_icon="\U0001f9e0", layout="wide")


# ---------------------------------------------------------------------------
# Cache the embedding model so it loads only once (not on every rerun)
# ---------------------------------------------------------------------------
@st.cache_resource(show_spinner="Loading embedding model...")
def load_model():
    from langchain_huggingface import HuggingFaceEmbeddings
    return HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


# ---------------------------------------------------------------------------
# Sidebar
# ---------------------------------------------------------------------------
st.sidebar.title("AI Explorer")
page = st.sidebar.radio(
    "Navigate",
    ["Home", "Embeddings", "Chunking", "Vector Search"]
)

# ---------------------------------------------------------------------------
# Home
# ---------------------------------------------------------------------------
if page == "Home":
    st.title("AI Explorer App")
    st.markdown("""
    ### What we'll explore:
    1. **Embeddings** -- How text becomes numbers
    2. **Chunking** -- How to split documents
    3. **Vector Search** -- How to search by meaning
    
    Use the sidebar to navigate.
    """)

# ---------------------------------------------------------------------------
# Embeddings page
# ---------------------------------------------------------------------------
elif page == "Embeddings":
    st.title("What Are Embeddings?")
    st.markdown("""
    An **embedding** turns text into a list of numbers (a vector) that captures its **meaning**.
    Similar texts get similar vectors.

    ```
    "The cat sat on the mat"    --> [0.12, -0.45, 0.78, ...] (384 numbers)
    "A kitten rested on a rug"  --> [0.11, -0.43, 0.76, ...] (similar!)
    "Stock prices rose today"   --> [-0.67, 0.22, -0.11, ...] (very different)
    ```
    """)

    st.divider()
    st.subheader("Try it: Compare two texts")

    col1, col2 = st.columns(2)
    with col1:
        text_a = st.text_input("Text A", "I love machine learning")
    with col2:
        text_b = st.text_input("Text B", "AI is fascinating")

    if st.button("Compare", type="primary"):
        model = load_model()

        # Embed both texts
        vectors = model.embed_documents([text_a, text_b])
        vectors = np.array(vectors)

        # Compute cosine similarity
        from sklearn.metrics.pairwise import cosine_similarity
        sim = cosine_similarity(vectors)[0, 1]

        # Display similarity
        st.metric("Cosine Similarity", f"{sim:.3f}")

        if sim > 0.7:
            st.success("These texts are very similar in meaning!")
        elif sim > 0.4:
            st.info("These texts share some meaning.")
        else:
            st.warning("These texts are quite different.")

        # Show vector details
        st.write(f"Each text was converted to a vector of **{len(vectors[0])}** numbers.")
        with st.expander("See the raw vectors"):
            st.code(f"Text A: {vectors[0][:10].tolist()}... ({len(vectors[0])} total)")
            st.code(f"Text B: {vectors[1][:10].tolist()}... ({len(vectors[1])} total)")

# ---------------------------------------------------------------------------
# Placeholder pages
# ---------------------------------------------------------------------------
elif page == "Chunking":
    st.title("Chunking Strategies")
    st.write("Coming next...")

elif page == "Vector Search":
    st.title("Vector Search")
    st.write("Coming next...")

### Go check your browser!

Click **"Embeddings"** in the sidebar, type two sentences, and click **"Compare"**.

Try these pairs to see the difference:
- "I love dogs" vs "Puppies are great" (high similarity)
- "I love dogs" vs "The stock market crashed" (low similarity)
- "The weather is nice" vs "It is a beautiful day" (high similarity)

### What's new here?

| Concept | Why it matters |
|---------|---------------|
| `@st.cache_resource` | The embedding model takes a few seconds to load. This decorator ensures it loads only **once**, not on every rerun. Without it, the app would freeze every time you click a button. |
| `st.button()` | Returns `True` when clicked. The code inside the `if` block runs only when the button is pressed. |
| `st.metric()` | A styled number display, perfect for scores and KPIs. |
| `st.success/info/warning()` | Colored alert boxes that draw attention to results. |
| `st.expander()` | A collapsible section -- keeps the UI clean by hiding details. |
| `st.columns()` | Places widgets side by side instead of stacked vertically. |

---

## Step 4: Similarity Matrix with Plotly

Comparing two texts is nice, but what if we want to compare many texts at once? Let's add a **similarity matrix** -- a heatmap that shows how every text relates to every other text.

We will extend the Embeddings page with a second section that takes multiple texts and visualizes them with Plotly.

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np

st.set_page_config(page_title="AI Explorer", page_icon="\U0001f9e0", layout="wide")


@st.cache_resource(show_spinner="Loading embedding model...")
def load_model():
    from langchain_huggingface import HuggingFaceEmbeddings
    return HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


st.sidebar.title("AI Explorer")
page = st.sidebar.radio(
    "Navigate",
    ["Home", "Embeddings", "Chunking", "Vector Search"]
)

if page == "Home":
    st.title("AI Explorer App")
    st.markdown("""
    ### What we'll explore:
    1. **Embeddings** -- How text becomes numbers
    2. **Chunking** -- How to split documents
    3. **Vector Search** -- How to search by meaning
    
    Use the sidebar to navigate.
    """)

elif page == "Embeddings":
    st.title("What Are Embeddings?")
    st.markdown("""
    An **embedding** turns text into a list of numbers (a vector) that captures its **meaning**.
    Similar texts get similar vectors.
    """)

    st.divider()

    # ----- Part 1: Compare two texts -----
    st.subheader("Part 1: Compare two texts")

    col1, col2 = st.columns(2)
    with col1:
        text_a = st.text_input("Text A", "I love machine learning")
    with col2:
        text_b = st.text_input("Text B", "AI is fascinating")

    if st.button("Compare", type="primary"):
        model = load_model()
        vectors = np.array(model.embed_documents([text_a, text_b]))

        from sklearn.metrics.pairwise import cosine_similarity
        sim = cosine_similarity(vectors)[0, 1]

        st.metric("Cosine Similarity", f"{sim:.3f}")

        if sim > 0.7:
            st.success("These texts are very similar in meaning!")
        elif sim > 0.4:
            st.info("These texts share some meaning.")
        else:
            st.warning("These texts are quite different.")

        with st.expander("See the raw vectors"):
            st.code(f"Text A: {vectors[0][:10].tolist()}... ({len(vectors[0])} dims)")
            st.code(f"Text B: {vectors[1][:10].tolist()}... ({len(vectors[1])} dims)")

    st.divider()

    # ----- Part 2: Similarity matrix -----
    st.subheader("Part 2: Similarity matrix")
    st.markdown("Enter multiple texts (one per line) to see how they all relate to each other.")

    extra_texts = st.text_area(
        "Texts to compare (one per line)",
        "I love machine learning\nAI is fascinating\n"
        "The weather is sunny today\nNeural networks learn from data\n"
        "I need to buy groceries",
        height=150
    )

    if st.button("Build Similarity Matrix", type="primary"):
        model = load_model()
        all_texts = [t.strip() for t in extra_texts.strip().split("\n") if t.strip()]

        if len(all_texts) < 2:
            st.error("Please enter at least 2 texts.")
        else:
            vectors = np.array(model.embed_documents(all_texts))

            from sklearn.metrics.pairwise import cosine_similarity
            sim_matrix = cosine_similarity(vectors)

            import plotly.express as px
            labels = [t[:40] + ("..." if len(t) > 40 else "") for t in all_texts]
            fig = px.imshow(
                sim_matrix,
                x=labels, y=labels,
                text_auto=".2f",
                color_continuous_scale="Blues",
                title="Cosine Similarity Matrix",
                aspect="auto"
            )
            fig.update_layout(height=500)
            st.plotly_chart(fig, use_container_width=True)

            st.info(f"Embedded {len(all_texts)} texts into {vectors.shape[1]}-dimensional vectors.")

elif page == "Chunking":
    st.title("Chunking Strategies")
    st.write("Coming next...")

elif page == "Vector Search":
    st.title("Vector Search")
    st.write("Coming next...")

### Go check your browser!

On the Embeddings page, scroll down to **Part 2** and click **"Build Similarity Matrix"**. You should see an interactive heatmap.

Try adding your own texts. Notice how semantically related texts cluster together with high similarity scores (close to 1.0) while unrelated texts show low scores.

### What's new here?

| Function | What it does |
|----------|-------------|
| `st.text_area()` | A multi-line text input (vs. `st.text_input()` for single line) |
| `st.plotly_chart()` | Renders an interactive Plotly chart. Users can hover, zoom, and pan. |
| `st.error()` | A red alert box for error messages |

**Key insight:** Plotly charts in Streamlit are fully interactive -- students can hover over cells to see exact similarity values. This is much better than a static matplotlib image.

---

## Step 5: Building the Chunking Strategies Page

Embedding models have a maximum input length. Real documents (articles, books, reports) are much longer than this limit. The solution is **chunking** -- splitting documents into smaller pieces before embedding them.

We will build an interactive page that demonstrates three chunking strategies:

1. **Fixed-Size** -- Split every N characters (simple but may break mid-sentence)
2. **Sentence-Based** -- Split at sentence boundaries (respects grammar)
3. **LangChain Recursive** -- Smart splitting that tries multiple separators (paragraphs, sentences, words)

**New Streamlit concepts:**
- `st.tabs()` -- Tabbed content areas
- `st.slider()` -- A draggable slider for numeric input

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import re

st.set_page_config(page_title="AI Explorer", page_icon="\U0001f9e0", layout="wide")


@st.cache_resource(show_spinner="Loading embedding model...")
def load_model():
    from langchain_huggingface import HuggingFaceEmbeddings
    return HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


# ---------------------------------------------------------------------------
# Sample text for the chunking demo
# ---------------------------------------------------------------------------
LONG_SAMPLE_TEXT = """Artificial Intelligence (AI) is a broad field of computer science focused on \
creating intelligent machines capable of performing tasks that typically require \
human intelligence. These tasks include learning from experience, understanding \
natural language, recognizing patterns, making decisions, and solving complex \
problems. The history of AI dates back to the 1950s when pioneers like Alan \
Turing proposed the famous Turing Test as a measure of machine intelligence.

Machine learning, a subset of AI, focuses on algorithms that improve through \
experience. Instead of being explicitly programmed, these systems learn patterns \
from data. There are three main types: supervised learning (learning from labeled \
examples), unsupervised learning (finding hidden patterns), and reinforcement \
learning (learning through trial and error). Deep learning, which uses neural \
networks with many layers, has achieved remarkable results in image recognition \
and natural language processing.

Natural Language Processing (NLP) is the branch of AI dealing with the \
interaction between computers and human language. Modern NLP relies heavily on \
transformer models, which use attention mechanisms to understand context and \
relationships between words. Embeddings, which convert text into numerical \
vectors, are fundamental to how these models represent meaning. This is what \
powers search engines, chatbots, and translation systems today."""


# ---------------------------------------------------------------------------
# Sidebar
# ---------------------------------------------------------------------------
st.sidebar.title("AI Explorer")
page = st.sidebar.radio(
    "Navigate",
    ["Home", "Embeddings", "Chunking", "Vector Search"]
)

# =========================================================================
# HOME
# =========================================================================
if page == "Home":
    st.title("AI Explorer App")
    st.markdown("""
    ### What we'll explore:
    1. **Embeddings** -- How text becomes numbers
    2. **Chunking** -- How to split documents
    3. **Vector Search** -- How to search by meaning
    
    Use the sidebar to navigate.
    """)

# =========================================================================
# EMBEDDINGS
# =========================================================================
elif page == "Embeddings":
    st.title("What Are Embeddings?")
    st.markdown("""
    An **embedding** turns text into a list of numbers (a vector) that captures its **meaning**.
    Similar texts get similar vectors.
    """)

    st.divider()
    st.subheader("Part 1: Compare two texts")

    col1, col2 = st.columns(2)
    with col1:
        text_a = st.text_input("Text A", "I love machine learning")
    with col2:
        text_b = st.text_input("Text B", "AI is fascinating")

    if st.button("Compare", type="primary"):
        model = load_model()
        vectors = np.array(model.embed_documents([text_a, text_b]))
        from sklearn.metrics.pairwise import cosine_similarity
        sim = cosine_similarity(vectors)[0, 1]

        st.metric("Cosine Similarity", f"{sim:.3f}")
        if sim > 0.7:
            st.success("These texts are very similar in meaning!")
        elif sim > 0.4:
            st.info("These texts share some meaning.")
        else:
            st.warning("These texts are quite different.")

        with st.expander("See the raw vectors"):
            st.code(f"Text A: {vectors[0][:10].tolist()}... ({len(vectors[0])} dims)")
            st.code(f"Text B: {vectors[1][:10].tolist()}... ({len(vectors[1])} dims)")

    st.divider()
    st.subheader("Part 2: Similarity matrix")

    extra_texts = st.text_area(
        "Texts to compare (one per line)",
        "I love machine learning\nAI is fascinating\n"
        "The weather is sunny today\nNeural networks learn from data\n"
        "I need to buy groceries",
        height=150
    )

    if st.button("Build Similarity Matrix", type="primary"):
        model = load_model()
        all_texts = [t.strip() for t in extra_texts.strip().split("\n") if t.strip()]

        if len(all_texts) < 2:
            st.error("Please enter at least 2 texts.")
        else:
            vectors = np.array(model.embed_documents(all_texts))
            from sklearn.metrics.pairwise import cosine_similarity
            sim_matrix = cosine_similarity(vectors)

            import plotly.express as px
            labels = [t[:40] + ("..." if len(t) > 40 else "") for t in all_texts]
            fig = px.imshow(
                sim_matrix, x=labels, y=labels,
                text_auto=".2f", color_continuous_scale="Blues",
                title="Cosine Similarity Matrix", aspect="auto"
            )
            fig.update_layout(height=500)
            st.plotly_chart(fig, use_container_width=True)

# =========================================================================
# CHUNKING
# =========================================================================
elif page == "Chunking":
    st.title("Chunking Strategies")
    st.markdown("""
    Real documents are too long for embedding models. We need to split them into
    **chunks** -- smaller pieces that can each be converted to a vector.

    The way you chunk affects search quality: too small and you lose context,
    too big and the meaning gets diluted.
    """)

    sample_text = st.text_area(
        "Paste a long text to chunk:",
        value=LONG_SAMPLE_TEXT,
        height=200
    )

    st.write(f"**Total length:** {len(sample_text)} characters")
    st.divider()

    tab1, tab2, tab3 = st.tabs(["Fixed-Size", "Sentence-Based", "LangChain Recursive"])

    # --- Tab 1: Fixed-size chunking ---
    with tab1:
        st.markdown("""
        **Fixed-size chunking** splits text every N characters, regardless of content.
        Simple, but may cut words or sentences in half.
        """)
        chunk_size = st.slider("Chunk size (characters)", 50, 500, 200, key="fixed_size")
        overlap = st.slider("Overlap (characters)", 0, 100, 30, key="fixed_overlap")

        chunks = []
        start = 0
        while start < len(sample_text):
            chunks.append(sample_text[start:start + chunk_size])
            start += chunk_size - overlap
            if overlap >= chunk_size:
                break

        st.write(f"**{len(chunks)} chunks** created")
        for i, chunk in enumerate(chunks):
            st.text_area(
                f"Chunk {i + 1} ({len(chunk)} chars)",
                chunk, height=80, key=f"fixed_{i}", disabled=True
            )

    # --- Tab 2: Sentence-based chunking ---
    with tab2:
        st.markdown("""
        **Sentence-based chunking** splits at sentence boundaries (periods, question marks, etc.).
        Each chunk is a complete sentence, preserving grammatical structure.
        """)
        sentences = re.split(r'(?<=[.!?])\s+', sample_text.strip())
        sentences = [s.strip() for s in sentences if s.strip()]

        st.write(f"**{len(sentences)} sentences** found")
        for i, sent in enumerate(sentences):
            st.text_area(
                f"Sentence {i + 1} ({len(sent)} chars)",
                sent, height=60, key=f"sent_{i}", disabled=True
            )

    # --- Tab 3: LangChain recursive ---
    with tab3:
        st.markdown("""
        **LangChain's RecursiveCharacterTextSplitter** is the smartest approach.
        It tries to split at paragraph boundaries first, then sentences, then words,
        then characters -- keeping chunks as semantically coherent as possible.
        """)
        rc_size = st.slider("Chunk size", 50, 500, 200, key="rc_size")
        rc_overlap = st.slider("Overlap", 0, 100, 30, key="rc_overlap")

        from langchain.text_splitter import RecursiveCharacterTextSplitter
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=rc_size,
            chunk_overlap=rc_overlap
        )
        rc_chunks = splitter.split_text(sample_text)

        st.write(f"**{len(rc_chunks)} chunks** created")
        for i, chunk in enumerate(rc_chunks):
            st.text_area(
                f"Chunk {i + 1} ({len(chunk)} chars)",
                chunk, height=80, key=f"rc_{i}", disabled=True
            )

# =========================================================================
# VECTOR SEARCH (placeholder)
# =========================================================================
elif page == "Vector Search":
    st.title("Vector Search")
    st.write("Coming next...")

### Go check your browser!

Click **"Chunking"** in the sidebar. You should see three tabs. Try each one:

1. **Fixed-Size** -- Drag the sliders to change chunk size and overlap. Notice how some chunks cut mid-word.
2. **Sentence-Based** -- See how each chunk is a clean, complete sentence.
3. **LangChain Recursive** -- The gold standard. Compare how it splits vs. fixed-size at the same chunk size.

### Experiment!

- Set fixed-size to 100 characters with 0 overlap. Look at the chunk boundaries -- are they clean?
- Now try LangChain Recursive at 100 characters. Notice how it finds better split points.
- Try a very small chunk size (50). What happens?

### What's new here?

| Function | What it does |
|----------|-------------|
| `st.tabs()` | Creates tabbed content areas. Each tab is a context manager (`with tab1:`) |
| `st.slider()` | A draggable slider. Returns the current value as a number. |
| `key="..."` | Every widget needs a unique key if you have multiple of the same type |
| `disabled=True` | Makes a text area read-only (for display purposes) |

---

## Step 6: Building the Vector Search Page

This is where everything comes together. A **vector database** stores embeddings and lets you search by meaning, not just keywords.

We will use **ChromaDB**, an open-source vector database that runs locally (no API keys needed).

**How it works:**
1. Take a collection of documents
2. Embed each document into a vector
3. Store the vectors in ChromaDB
4. When a user searches, embed the query and find the most similar document vectors

This is the core of **RAG (Retrieval-Augmented Generation)** -- the technique behind ChatGPT plugins, Bing Chat, and similar tools.

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import re

st.set_page_config(page_title="AI Explorer", page_icon="\U0001f9e0", layout="wide")


@st.cache_resource(show_spinner="Loading embedding model...")
def load_model():
    from langchain_huggingface import HuggingFaceEmbeddings
    return HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


# ---------------------------------------------------------------------------
# Sample text for chunking
# ---------------------------------------------------------------------------
LONG_SAMPLE_TEXT = """Artificial Intelligence (AI) is a broad field of computer science focused on \
creating intelligent machines capable of performing tasks that typically require \
human intelligence. These tasks include learning from experience, understanding \
natural language, recognizing patterns, making decisions, and solving complex \
problems. The history of AI dates back to the 1950s when pioneers like Alan \
Turing proposed the famous Turing Test as a measure of machine intelligence.

Machine learning, a subset of AI, focuses on algorithms that improve through \
experience. Instead of being explicitly programmed, these systems learn patterns \
from data. There are three main types: supervised learning (learning from labeled \
examples), unsupervised learning (finding hidden patterns), and reinforcement \
learning (learning through trial and error). Deep learning, which uses neural \
networks with many layers, has achieved remarkable results in image recognition \
and natural language processing.

Natural Language Processing (NLP) is the branch of AI dealing with the \
interaction between computers and human language. Modern NLP relies heavily on \
transformer models, which use attention mechanisms to understand context and \
relationships between words. Embeddings, which convert text into numerical \
vectors, are fundamental to how these models represent meaning. This is what \
powers search engines, chatbots, and translation systems today."""


# ---------------------------------------------------------------------------
# Sidebar
# ---------------------------------------------------------------------------
st.sidebar.title("AI Explorer")
page = st.sidebar.radio(
    "Navigate",
    ["Home", "Embeddings", "Chunking", "Vector Search"]
)

# =========================================================================
# HOME
# =========================================================================
if page == "Home":
    st.title("AI Explorer App")
    st.markdown("""
    ### What we'll explore:
    1. **Embeddings** -- How text becomes numbers
    2. **Chunking** -- How to split documents
    3. **Vector Search** -- How to search by meaning
    
    Use the sidebar to navigate.
    """)

# =========================================================================
# EMBEDDINGS
# =========================================================================
elif page == "Embeddings":
    st.title("What Are Embeddings?")
    st.markdown("""
    An **embedding** turns text into a list of numbers (a vector) that captures its **meaning**.
    Similar texts get similar vectors.
    """)

    st.divider()
    st.subheader("Part 1: Compare two texts")

    col1, col2 = st.columns(2)
    with col1:
        text_a = st.text_input("Text A", "I love machine learning")
    with col2:
        text_b = st.text_input("Text B", "AI is fascinating")

    if st.button("Compare", type="primary"):
        model = load_model()
        vectors = np.array(model.embed_documents([text_a, text_b]))
        from sklearn.metrics.pairwise import cosine_similarity
        sim = cosine_similarity(vectors)[0, 1]

        st.metric("Cosine Similarity", f"{sim:.3f}")
        if sim > 0.7:
            st.success("These texts are very similar in meaning!")
        elif sim > 0.4:
            st.info("These texts share some meaning.")
        else:
            st.warning("These texts are quite different.")

        with st.expander("See the raw vectors"):
            st.code(f"Text A: {vectors[0][:10].tolist()}... ({len(vectors[0])} dims)")
            st.code(f"Text B: {vectors[1][:10].tolist()}... ({len(vectors[1])} dims)")

    st.divider()
    st.subheader("Part 2: Similarity matrix")

    extra_texts = st.text_area(
        "Texts to compare (one per line)",
        "I love machine learning\nAI is fascinating\n"
        "The weather is sunny today\nNeural networks learn from data\n"
        "I need to buy groceries",
        height=150
    )

    if st.button("Build Similarity Matrix", type="primary"):
        model = load_model()
        all_texts = [t.strip() for t in extra_texts.strip().split("\n") if t.strip()]

        if len(all_texts) < 2:
            st.error("Please enter at least 2 texts.")
        else:
            vectors = np.array(model.embed_documents(all_texts))
            from sklearn.metrics.pairwise import cosine_similarity
            sim_matrix = cosine_similarity(vectors)

            import plotly.express as px
            labels = [t[:40] + ("..." if len(t) > 40 else "") for t in all_texts]
            fig = px.imshow(
                sim_matrix, x=labels, y=labels,
                text_auto=".2f", color_continuous_scale="Blues",
                title="Cosine Similarity Matrix", aspect="auto"
            )
            fig.update_layout(height=500)
            st.plotly_chart(fig, use_container_width=True)

# =========================================================================
# CHUNKING
# =========================================================================
elif page == "Chunking":
    st.title("Chunking Strategies")
    st.markdown("""
    Real documents are too long for embedding models. We need to split them into
    **chunks** -- smaller pieces that can each be converted to a vector.
    """)

    sample_text = st.text_area(
        "Paste a long text to chunk:",
        value=LONG_SAMPLE_TEXT,
        height=200
    )
    st.write(f"**Total length:** {len(sample_text)} characters")
    st.divider()

    tab1, tab2, tab3 = st.tabs(["Fixed-Size", "Sentence-Based", "LangChain Recursive"])

    with tab1:
        st.markdown("**Fixed-size** splits every N characters, regardless of content.")
        chunk_size = st.slider("Chunk size (characters)", 50, 500, 200, key="fixed_size")
        overlap = st.slider("Overlap (characters)", 0, 100, 30, key="fixed_overlap")

        chunks = []
        start = 0
        while start < len(sample_text):
            chunks.append(sample_text[start:start + chunk_size])
            start += chunk_size - overlap
            if overlap >= chunk_size:
                break

        st.write(f"**{len(chunks)} chunks** created")
        for i, chunk in enumerate(chunks):
            st.text_area(f"Chunk {i+1} ({len(chunk)} chars)", chunk,
                         height=80, key=f"fixed_{i}", disabled=True)

    with tab2:
        st.markdown("**Sentence-based** splits at sentence boundaries.")
        sentences = re.split(r'(?<=[.!?])\s+', sample_text.strip())
        sentences = [s.strip() for s in sentences if s.strip()]

        st.write(f"**{len(sentences)} sentences** found")
        for i, sent in enumerate(sentences):
            st.text_area(f"Sentence {i+1} ({len(sent)} chars)", sent,
                         height=60, key=f"sent_{i}", disabled=True)

    with tab3:
        st.markdown("**LangChain Recursive** tries paragraphs, then sentences, then words.")
        rc_size = st.slider("Chunk size", 50, 500, 200, key="rc_size")
        rc_overlap = st.slider("Overlap", 0, 100, 30, key="rc_overlap")

        from langchain.text_splitter import RecursiveCharacterTextSplitter
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=rc_size, chunk_overlap=rc_overlap
        )
        rc_chunks = splitter.split_text(sample_text)

        st.write(f"**{len(rc_chunks)} chunks** created")
        for i, chunk in enumerate(rc_chunks):
            st.text_area(f"Chunk {i+1} ({len(chunk)} chars)", chunk,
                         height=80, key=f"rc_{i}", disabled=True)

# =========================================================================
# VECTOR SEARCH
# =========================================================================
elif page == "Vector Search":
    st.title("Vector Search with ChromaDB")
    st.markdown("""
    A **vector database** stores embeddings and searches by **meaning**, not keywords.

    Traditional search: *"Does the document contain the exact words I typed?"*
    
    Vector search: *"Does the document mean something similar to what I typed?"*
    """)

    # Sample knowledge base
    documents = [
        "Black holes are regions of spacetime where gravity is so strong that nothing can escape.",
        "DNA carries the genetic instructions for the development and functioning of all living organisms.",
        "Machine learning algorithms learn patterns from data without being explicitly programmed.",
        "Quantum computers use qubits that can exist in multiple states simultaneously.",
        "Neural networks are computing systems inspired by the biological neurons in the human brain.",
        "Photosynthesis is the process by which plants convert sunlight into chemical energy.",
        "The theory of relativity describes the relationship between space, time, and gravity.",
        "Antibiotics are medicines that fight bacterial infections in humans and animals.",
        "Climate change refers to long-term shifts in global temperatures and weather patterns.",
        "The Internet connects billions of devices worldwide through a network of networks.",
    ]

    st.markdown("**Documents in our knowledge base:**")
    for i, doc in enumerate(documents, 1):
        st.markdown(f"{i}. {doc}")

    st.divider()

    query = st.text_input(
        "Search query",
        placeholder="e.g., 'How do computers learn?' or 'What powers plants?'"
    )
    num_results = st.slider("Number of results", 1, 5, 3, key="num_results")

    if st.button("Search", type="primary") and query:
        model = load_model()

        with st.spinner("Searching..."):
            from langchain_community.vectorstores import Chroma
            vectorstore = Chroma.from_texts(texts=documents, embedding=model)
            results = vectorstore.similarity_search_with_relevance_scores(
                query, k=num_results
            )

        st.subheader("Results (ranked by meaning similarity)")
        for i, (doc, score) in enumerate(results, 1):
            pct = max(0, score) * 100
            col_text, col_score = st.columns([4, 1])
            with col_text:
                st.markdown(f"**{i}.** {doc.page_content}")
            with col_score:
                st.metric("Relevance", f"{pct:.0f}%")
            st.progress(min(1.0, max(0, score)))

        st.divider()
        st.markdown("""
        **Notice:** The search finds relevant documents even if they don't share
        any exact words with your query. That is the power of vector search!
        
        Try searching for:
        - *"How do computers learn?"* -- finds machine learning and neural network docs
        - *"What makes plants grow?"* -- finds photosynthesis
        - *"Space and the universe"* -- finds black holes and relativity
        """)

### Go check your browser!

Click **"Vector Search"** in the sidebar and try searching with natural language queries.

**Key things to notice:**

- Search for "How do computers learn?" -- it finds the machine learning document even though the exact phrase does not appear in any document.
- Search for "What makes plants grow?" -- it finds photosynthesis.
- This is **semantic search** -- searching by meaning, not by keywords.

### What's new here?

| Concept | What it does |
|---------|-------------|
| `Chroma.from_texts()` | Creates a vector database from a list of strings. It automatically embeds each text using the model we provide. |
| `.similarity_search_with_relevance_scores()` | Searches the database and returns both the document and a relevance score (0 to 1). |
| `st.spinner()` | Shows a loading animation while code runs inside its `with` block. |
| `st.progress()` | A progress bar -- we use it to visually show relevance scores. |

---

## Step 7: Custom Styling

The app works, but let's make it look more polished with some custom CSS. Streamlit allows you to inject CSS using `st.markdown()` with `unsafe_allow_html=True`.

We will add:
- A dark background
- Gradient-colored headings
- Styled buttons
- Rounded metric cards

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import re

st.set_page_config(page_title="AI Explorer", page_icon="\U0001f9e0", layout="wide")

# ---------------------------------------------------------------------------
# Custom CSS -- makes the app look polished
# ---------------------------------------------------------------------------
st.markdown('''
<style>
    /* Dark background */
    .stApp {
        background-color: #0f0f1a;
    }

    /* Sidebar */
    section[data-testid="stSidebar"] {
        background-color: #0a0a14;
        border-right: 1px solid #1a1a2e;
    }

    /* Gradient headings */
    h1 {
        background: linear-gradient(120deg, #4285f4, #8b5cf6);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        font-weight: 800 !important;
    }

    /* Styled metric cards */
    div[data-testid="metric-container"] {
        background-color: #1a1a2e;
        border: 1px solid #2a2a4e;
        border-radius: 10px;
        padding: 15px;
    }

    /* Styled buttons */
    .stButton > button {
        background: linear-gradient(135deg, #4285f4, #8b5cf6);
        color: white;
        border: none;
        border-radius: 8px;
        font-weight: 600;
    }

    /* Rounded code blocks */
    .stCodeBlock {
        border-radius: 10px;
    }

    /* Alert boxes */
    .stAlert {
        border-radius: 10px;
    }

    /* Tab styling */
    .stTabs [data-baseweb="tab-list"] {
        gap: 8px;
    }
    .stTabs [data-baseweb="tab"] {
        border-radius: 8px;
        padding: 8px 16px;
    }
</style>
''', unsafe_allow_html=True)


@st.cache_resource(show_spinner="Loading embedding model...")
def load_model():
    from langchain_huggingface import HuggingFaceEmbeddings
    return HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


LONG_SAMPLE_TEXT = """Artificial Intelligence (AI) is a broad field of computer science focused on \
creating intelligent machines capable of performing tasks that typically require \
human intelligence. These tasks include learning from experience, understanding \
natural language, recognizing patterns, making decisions, and solving complex \
problems. The history of AI dates back to the 1950s when pioneers like Alan \
Turing proposed the famous Turing Test as a measure of machine intelligence.

Machine learning, a subset of AI, focuses on algorithms that improve through \
experience. Instead of being explicitly programmed, these systems learn patterns \
from data. There are three main types: supervised learning (learning from labeled \
examples), unsupervised learning (finding hidden patterns), and reinforcement \
learning (learning through trial and error). Deep learning, which uses neural \
networks with many layers, has achieved remarkable results in image recognition \
and natural language processing.

Natural Language Processing (NLP) is the branch of AI dealing with the \
interaction between computers and human language. Modern NLP relies heavily on \
transformer models, which use attention mechanisms to understand context and \
relationships between words. Embeddings, which convert text into numerical \
vectors, are fundamental to how these models represent meaning. This is what \
powers search engines, chatbots, and translation systems today."""


st.sidebar.title("AI Explorer")
page = st.sidebar.radio(
    "Navigate",
    ["Home", "Embeddings", "Chunking", "Vector Search"]
)

# =========================================================================
# HOME
# =========================================================================
if page == "Home":
    st.title("AI Explorer App")
    st.markdown("""
    ### What we'll explore:
    1. **Embeddings** -- How text becomes numbers
    2. **Chunking** -- How to split documents
    3. **Vector Search** -- How to search by meaning
    
    Use the sidebar to navigate.
    """)

# =========================================================================
# EMBEDDINGS
# =========================================================================
elif page == "Embeddings":
    st.title("What Are Embeddings?")
    st.markdown("""
    An **embedding** turns text into a list of numbers (a vector) that captures its **meaning**.
    Similar texts get similar vectors.
    """)

    st.divider()
    st.subheader("Part 1: Compare two texts")

    col1, col2 = st.columns(2)
    with col1:
        text_a = st.text_input("Text A", "I love machine learning")
    with col2:
        text_b = st.text_input("Text B", "AI is fascinating")

    if st.button("Compare", type="primary"):
        model = load_model()
        vectors = np.array(model.embed_documents([text_a, text_b]))
        from sklearn.metrics.pairwise import cosine_similarity
        sim = cosine_similarity(vectors)[0, 1]

        st.metric("Cosine Similarity", f"{sim:.3f}")
        if sim > 0.7:
            st.success("These texts are very similar in meaning!")
        elif sim > 0.4:
            st.info("These texts share some meaning.")
        else:
            st.warning("These texts are quite different.")

        with st.expander("See the raw vectors"):
            st.code(f"Text A: {vectors[0][:10].tolist()}... ({len(vectors[0])} dims)")
            st.code(f"Text B: {vectors[1][:10].tolist()}... ({len(vectors[1])} dims)")

    st.divider()
    st.subheader("Part 2: Similarity matrix")

    extra_texts = st.text_area(
        "Texts to compare (one per line)",
        "I love machine learning\nAI is fascinating\n"
        "The weather is sunny today\nNeural networks learn from data\n"
        "I need to buy groceries",
        height=150
    )

    if st.button("Build Similarity Matrix", type="primary"):
        model = load_model()
        all_texts = [t.strip() for t in extra_texts.strip().split("\n") if t.strip()]

        if len(all_texts) < 2:
            st.error("Please enter at least 2 texts.")
        else:
            vectors = np.array(model.embed_documents(all_texts))
            from sklearn.metrics.pairwise import cosine_similarity
            sim_matrix = cosine_similarity(vectors)

            import plotly.express as px
            labels = [t[:40] + ("..." if len(t) > 40 else "") for t in all_texts]
            fig = px.imshow(
                sim_matrix, x=labels, y=labels,
                text_auto=".2f", color_continuous_scale="Blues",
                title="Cosine Similarity Matrix", aspect="auto"
            )
            fig.update_layout(height=500)
            st.plotly_chart(fig, use_container_width=True)

# =========================================================================
# CHUNKING
# =========================================================================
elif page == "Chunking":
    st.title("Chunking Strategies")
    st.markdown("""
    Real documents are too long for embedding models. We need to split them into
    **chunks** -- smaller pieces that can each be converted to a vector.
    """)

    sample_text = st.text_area("Paste a long text to chunk:",
                               value=LONG_SAMPLE_TEXT, height=200)
    st.write(f"**Total length:** {len(sample_text)} characters")
    st.divider()

    tab1, tab2, tab3 = st.tabs(["Fixed-Size", "Sentence-Based", "LangChain Recursive"])

    with tab1:
        st.markdown("**Fixed-size** splits every N characters, regardless of content.")
        chunk_size = st.slider("Chunk size (characters)", 50, 500, 200, key="fixed_size")
        overlap = st.slider("Overlap (characters)", 0, 100, 30, key="fixed_overlap")
        chunks = []
        start = 0
        while start < len(sample_text):
            chunks.append(sample_text[start:start + chunk_size])
            start += chunk_size - overlap
            if overlap >= chunk_size:
                break
        st.write(f"**{len(chunks)} chunks** created")
        for i, chunk in enumerate(chunks):
            st.text_area(f"Chunk {i+1} ({len(chunk)} chars)", chunk,
                         height=80, key=f"fixed_{i}", disabled=True)

    with tab2:
        st.markdown("**Sentence-based** splits at sentence boundaries.")
        sentences = re.split(r'(?<=[.!?])\s+', sample_text.strip())
        sentences = [s.strip() for s in sentences if s.strip()]
        st.write(f"**{len(sentences)} sentences** found")
        for i, sent in enumerate(sentences):
            st.text_area(f"Sentence {i+1} ({len(sent)} chars)", sent,
                         height=60, key=f"sent_{i}", disabled=True)

    with tab3:
        st.markdown("**LangChain Recursive** tries paragraphs, then sentences, then words.")
        rc_size = st.slider("Chunk size", 50, 500, 200, key="rc_size")
        rc_overlap = st.slider("Overlap", 0, 100, 30, key="rc_overlap")
        from langchain.text_splitter import RecursiveCharacterTextSplitter
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=rc_size, chunk_overlap=rc_overlap
        )
        rc_chunks = splitter.split_text(sample_text)
        st.write(f"**{len(rc_chunks)} chunks** created")
        for i, chunk in enumerate(rc_chunks):
            st.text_area(f"Chunk {i+1} ({len(chunk)} chars)", chunk,
                         height=80, key=f"rc_{i}", disabled=True)

# =========================================================================
# VECTOR SEARCH
# =========================================================================
elif page == "Vector Search":
    st.title("Vector Search with ChromaDB")
    st.markdown("""
    A **vector database** stores embeddings and searches by **meaning**, not keywords.
    """)

    documents = [
        "Black holes are regions of spacetime where gravity is so strong that nothing can escape.",
        "DNA carries the genetic instructions for the development and functioning of all living organisms.",
        "Machine learning algorithms learn patterns from data without being explicitly programmed.",
        "Quantum computers use qubits that can exist in multiple states simultaneously.",
        "Neural networks are computing systems inspired by the biological neurons in the human brain.",
        "Photosynthesis is the process by which plants convert sunlight into chemical energy.",
        "The theory of relativity describes the relationship between space, time, and gravity.",
        "Antibiotics are medicines that fight bacterial infections in humans and animals.",
        "Climate change refers to long-term shifts in global temperatures and weather patterns.",
        "The Internet connects billions of devices worldwide through a network of networks.",
    ]

    st.markdown("**Documents in our knowledge base:**")
    for i, doc in enumerate(documents, 1):
        st.markdown(f"{i}. {doc}")

    st.divider()

    query = st.text_input("Search query",
                          placeholder="e.g., 'How do computers learn?'")
    num_results = st.slider("Number of results", 1, 5, 3, key="num_results")

    if st.button("Search", type="primary") and query:
        model = load_model()

        with st.spinner("Searching..."):
            from langchain_community.vectorstores import Chroma
            vectorstore = Chroma.from_texts(texts=documents, embedding=model)
            results = vectorstore.similarity_search_with_relevance_scores(
                query, k=num_results
            )

        st.subheader("Results (ranked by meaning similarity)")
        for i, (doc, score) in enumerate(results, 1):
            pct = max(0, score) * 100
            col_text, col_score = st.columns([4, 1])
            with col_text:
                st.markdown(f"**{i}.** {doc.page_content}")
            with col_score:
                st.metric("Relevance", f"{pct:.0f}%")
            st.progress(min(1.0, max(0, score)))

        st.divider()
        st.info("""
        **Try these queries:**
        - "How do computers learn?" -- finds ML and neural network docs
        - "What makes plants grow?" -- finds photosynthesis
        - "Space and the universe" -- finds black holes and relativity
        - "Fighting disease" -- finds antibiotics
        """)

### Go check your browser!

The app should now have a dark theme with gradient headings and styled buttons.

### What's new here?

We added a `<style>` block at the top using `st.markdown(..., unsafe_allow_html=True)`. This is standard CSS injected into the page.

**Key CSS tricks:**
- `background: linear-gradient(...)` -- creates smooth color transitions
- `-webkit-background-clip: text` -- applies the gradient to text instead of background
- `border-radius` -- rounds corners
- We target Streamlit's internal CSS classes (like `div[data-testid="metric-container"]`) to style specific components

> **Note:** Custom CSS in Streamlit can break between versions since the internal class names may change. Use it for polish, not for critical functionality.

---

## Step 8: Sharing Your App on GitHub

To share your app with the world (or deploy it to the cloud), you need to push it to GitHub.

### 8.1 Create a requirements.txt

Streamlit Cloud (and other platforms) need to know which packages to install.

In [ ]:
%%writefile requirements.txt
streamlit>=1.30.0
langchain>=0.2.0
langchain-community>=0.2.0
langchain-huggingface>=0.0.3
chromadb>=0.4.22
sentence-transformers>=2.2.2
scikit-learn>=1.3.0
numpy>=1.24.0
pandas>=2.0.0
plotly>=5.18.0

### 8.2 Push to GitHub

Run these commands to initialize a git repository and push to GitHub.

> **Before running:** Replace `YOUR_USERNAME` with your GitHub username. You also need to create an empty repository on GitHub first (github.com/new).

In [ ]:
# Uncomment and run these commands after creating a GitHub repo:

# !git init
# !git add app.py requirements.txt
# !git commit -m "Initial Streamlit app - AI Explorer"
# !git branch -M main
# !git remote add origin https://github.com/YOUR_USERNAME/ai-explorer-app.git
# !git push -u origin main

### 8.3 Understanding the git workflow

Here is what each command does:

| Command | What it does |
|---------|-------------|
| `git init` | Creates a new git repository in the current folder |
| `git add app.py requirements.txt` | Stages the files you want to track |
| `git commit -m "..."` | Saves a snapshot of the staged files with a message |
| `git branch -M main` | Renames the default branch to `main` |
| `git remote add origin <url>` | Connects your local repo to GitHub |
| `git push -u origin main` | Uploads your code to GitHub |

After pushing, your code will be visible at `https://github.com/YOUR_USERNAME/ai-explorer-app`.

---

## Step 9: Putting It All Together -- The Complete App

Here is the final, complete version of `app.py` with all pages, styling, and features combined. This is what you should have at this point.

If anything went wrong in previous steps, you can run this cell to get the complete working app.

In [ ]:
%%writefile app.py
"""
AI Explorer -- Interactive Workshop App
FAMNIT AI Course - Day 2

An interactive Streamlit application for exploring:
- Embeddings and semantic similarity
- Document chunking strategies
- Vector search with ChromaDB
"""

import streamlit as st
import numpy as np
import re

# =========================================================================
# PAGE CONFIG (must be first Streamlit command)
# =========================================================================
st.set_page_config(page_title="AI Explorer", page_icon="\U0001f9e0", layout="wide")

# =========================================================================
# CUSTOM CSS
# =========================================================================
st.markdown('''
<style>
    .stApp {
        background-color: #0f0f1a;
    }
    section[data-testid="stSidebar"] {
        background-color: #0a0a14;
        border-right: 1px solid #1a1a2e;
    }
    h1 {
        background: linear-gradient(120deg, #4285f4, #8b5cf6);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        font-weight: 800 !important;
    }
    div[data-testid="metric-container"] {
        background-color: #1a1a2e;
        border: 1px solid #2a2a4e;
        border-radius: 10px;
        padding: 15px;
    }
    .stButton > button {
        background: linear-gradient(135deg, #4285f4, #8b5cf6);
        color: white;
        border: none;
        border-radius: 8px;
        font-weight: 600;
    }
    .stCodeBlock { border-radius: 10px; }
    .stAlert { border-radius: 10px; }
    .stTabs [data-baseweb="tab-list"] { gap: 8px; }
    .stTabs [data-baseweb="tab"] { border-radius: 8px; padding: 8px 16px; }
    .workshop-card {
        background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
        border: 1px solid #2a2a4e;
        border-radius: 12px;
        padding: 20px;
        margin: 10px 0;
        text-align: center;
    }
</style>
''', unsafe_allow_html=True)

# =========================================================================
# CACHED RESOURCES
# =========================================================================

@st.cache_resource(show_spinner="Loading embedding model...")
def load_model():
    from langchain_huggingface import HuggingFaceEmbeddings
    return HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


# =========================================================================
# SAMPLE TEXT
# =========================================================================
LONG_SAMPLE_TEXT = """Artificial Intelligence (AI) is a broad field of computer science focused on \
creating intelligent machines capable of performing tasks that typically require \
human intelligence. These tasks include learning from experience, understanding \
natural language, recognizing patterns, making decisions, and solving complex \
problems. The history of AI dates back to the 1950s when pioneers like Alan \
Turing proposed the famous Turing Test as a measure of machine intelligence.

Machine learning, a subset of AI, focuses on algorithms that improve through \
experience. Instead of being explicitly programmed, these systems learn patterns \
from data. There are three main types: supervised learning (learning from labeled \
examples), unsupervised learning (finding hidden patterns), and reinforcement \
learning (learning through trial and error). Deep learning, which uses neural \
networks with many layers, has achieved remarkable results in image recognition \
and natural language processing.

Natural Language Processing (NLP) is the branch of AI dealing with the \
interaction between computers and human language. Modern NLP relies heavily on \
transformer models, which use attention mechanisms to understand context and \
relationships between words. Embeddings, which convert text into numerical \
vectors, are fundamental to how these models represent meaning. This is what \
powers search engines, chatbots, and translation systems today."""


# =========================================================================
# SIDEBAR
# =========================================================================
st.sidebar.title("AI Explorer")
page = st.sidebar.radio(
    "Navigate",
    ["Home", "Embeddings", "Chunking", "Vector Search"]
)

# =========================================================================
# HOME PAGE
# =========================================================================
if page == "Home":
    st.title("AI Explorer")
    st.subheader("Interactive Workshop -- Day 2")

    st.markdown("---")

    # Visual workflow diagram
    c1, c2, c3, c4 = st.columns(4)
    for col, (title, icon, desc) in zip(
        [c1, c2, c3, c4],
        [
            ("Raw Text", "\U0001f4c4", "Documents and articles"),
            ("Chunking", "\u2702\ufe0f", "Split into pieces"),
            ("Embeddings", "\U0001f522", "Text becomes vectors"),
            ("Vector Search", "\U0001f50d", "Search by meaning"),
        ]
    ):
        with col:
            st.markdown(f'''
            <div class="workshop-card">
            <h4>{title}</h4>
            <p style="font-size:2em;">{icon}</p>
            <p>{desc}</p>
            </div>
            ''', unsafe_allow_html=True)

    st.markdown("---")

    st.markdown("""
    ### Learning Objectives

    By the end of this workshop you will be able to:

    - **Explain** what embeddings are and why they capture semantic meaning
    - **Compare** different chunking strategies and know when to use each
    - **Build** a vector database with ChromaDB and perform semantic search
    - **Understand** how RAG (Retrieval-Augmented Generation) works under the hood

    ### Key Idea

    > **Embeddings** convert text into numbers (vectors) that capture *meaning*.
    > Similar texts get similar vectors. This is the foundation of modern AI
    > search, recommendation systems, and RAG.

    Click **"Embeddings"** in the sidebar to begin.
    """)

# =========================================================================
# EMBEDDINGS PAGE
# =========================================================================
elif page == "Embeddings":
    st.title("What Are Embeddings?")
    st.markdown("""
    An **embedding** turns text into a list of numbers (a vector) that captures its **meaning**.
    Similar texts get similar vectors.

    ```
    "The cat sat on the mat"    --> [0.12, -0.45, 0.78, ...] (384 numbers)
    "A kitten rested on a rug"  --> [0.11, -0.43, 0.76, ...] (similar!)
    "Stock prices rose today"   --> [-0.67, 0.22, -0.11, ...] (very different)
    ```
    """)

    st.divider()

    # --- Part 1: Compare two texts ---
    st.subheader("Part 1: Compare two texts")

    col1, col2 = st.columns(2)
    with col1:
        text_a = st.text_input("Text A", "I love machine learning")
    with col2:
        text_b = st.text_input("Text B", "AI is fascinating")

    if st.button("Compare", type="primary"):
        model = load_model()
        vectors = np.array(model.embed_documents([text_a, text_b]))

        from sklearn.metrics.pairwise import cosine_similarity
        sim = cosine_similarity(vectors)[0, 1]

        col_metric, col_interp = st.columns(2)
        with col_metric:
            st.metric("Cosine Similarity", f"{sim:.3f}")
        with col_interp:
            if sim > 0.7:
                st.success("These texts are very similar in meaning!")
            elif sim > 0.4:
                st.info("These texts share some meaning.")
            else:
                st.warning("These texts are quite different.")

        st.write(f"Each text was converted to a vector of **{len(vectors[0])}** numbers.")

        with st.expander("See the raw vectors"):
            st.code(f"Text A: {vectors[0][:10].tolist()}... ({len(vectors[0])} dims)")
            st.code(f"Text B: {vectors[1][:10].tolist()}... ({len(vectors[1])} dims)")

    st.divider()

    # --- Part 2: Similarity matrix ---
    st.subheader("Part 2: Similarity matrix")
    st.markdown("Enter multiple texts (one per line) to see how they all relate.")

    extra_texts = st.text_area(
        "Texts to compare (one per line)",
        "I love machine learning\nAI is fascinating\n"
        "The weather is sunny today\nNeural networks learn from data\n"
        "I need to buy groceries\nIt is a beautiful warm day outside",
        height=150
    )

    if st.button("Build Similarity Matrix", type="primary"):
        model = load_model()
        all_texts = [t.strip() for t in extra_texts.strip().split("\n") if t.strip()]

        if len(all_texts) < 2:
            st.error("Please enter at least 2 texts.")
        else:
            with st.spinner("Embedding texts..."):
                vectors = np.array(model.embed_documents(all_texts))
                from sklearn.metrics.pairwise import cosine_similarity
                sim_matrix = cosine_similarity(vectors)

            import plotly.express as px
            labels = [t[:40] + ("..." if len(t) > 40 else "") for t in all_texts]
            fig = px.imshow(
                sim_matrix, x=labels, y=labels,
                text_auto=".2f", color_continuous_scale="Blues",
                title="Cosine Similarity Matrix", aspect="auto"
            )
            fig.update_layout(height=500)
            st.plotly_chart(fig, use_container_width=True)

            st.info(f"Embedded {len(all_texts)} texts into {vectors.shape[1]}-dimensional vectors.")

# =========================================================================
# CHUNKING PAGE
# =========================================================================
elif page == "Chunking":
    st.title("Chunking Strategies")
    st.markdown("""
    Real documents are too long for embedding models. We need to split them into
    **chunks** -- smaller pieces that can each be converted to a vector.

    The way you chunk affects search quality:
    - **Too small** = chunks lose context
    - **Too big** = meaning gets diluted
    - **Just right** = each chunk captures one coherent idea
    """)

    sample_text = st.text_area("Paste a long text to chunk:",
                               value=LONG_SAMPLE_TEXT, height=200)
    st.write(f"**Total length:** {len(sample_text)} characters")
    st.divider()

    tab1, tab2, tab3 = st.tabs(
        ["Fixed-Size Chunking", "Sentence-Based Chunking", "LangChain Recursive"]
    )

    # --- Fixed-size ---
    with tab1:
        st.markdown("""
        **Fixed-size chunking** splits text every N characters, regardless of content.
        Simple but may cut words or sentences in half.
        """)
        chunk_size = st.slider("Chunk size (characters)", 50, 500, 200, key="fixed_size")
        overlap = st.slider("Overlap (characters)", 0, 100, 30, key="fixed_overlap")

        chunks = []
        start = 0
        while start < len(sample_text):
            chunks.append(sample_text[start:start + chunk_size])
            start += chunk_size - overlap
            if overlap >= chunk_size:
                break

        st.write(f"**{len(chunks)} chunks** created")
        for i, chunk in enumerate(chunks):
            st.text_area(f"Chunk {i+1} ({len(chunk)} chars)", chunk,
                         height=80, key=f"fixed_{i}", disabled=True)

    # --- Sentence-based ---
    with tab2:
        st.markdown("""
        **Sentence-based chunking** splits at sentence boundaries (periods, etc.).
        Each chunk is a complete sentence.
        """)
        sentences = re.split(r'(?<=[.!?])\s+', sample_text.strip())
        sentences = [s.strip() for s in sentences if s.strip()]

        st.write(f"**{len(sentences)} sentences** found")
        for i, sent in enumerate(sentences):
            st.text_area(f"Sentence {i+1} ({len(sent)} chars)", sent,
                         height=60, key=f"sent_{i}", disabled=True)

    # --- LangChain Recursive ---
    with tab3:
        st.markdown("""
        **LangChain RecursiveCharacterTextSplitter** is the smartest approach.
        It tries to split at paragraph boundaries first, then sentences, then words.
        """)
        rc_size = st.slider("Chunk size", 50, 500, 200, key="rc_size")
        rc_overlap = st.slider("Overlap", 0, 100, 30, key="rc_overlap")

        from langchain.text_splitter import RecursiveCharacterTextSplitter
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=rc_size, chunk_overlap=rc_overlap
        )
        rc_chunks = splitter.split_text(sample_text)

        st.write(f"**{len(rc_chunks)} chunks** created")
        for i, chunk in enumerate(rc_chunks):
            st.text_area(f"Chunk {i+1} ({len(chunk)} chars)", chunk,
                         height=80, key=f"rc_{i}", disabled=True)

# =========================================================================
# VECTOR SEARCH PAGE
# =========================================================================
elif page == "Vector Search":
    st.title("Vector Search with ChromaDB")
    st.markdown("""
    A **vector database** stores embeddings and searches by **meaning**, not keywords.

    **Traditional search:** *"Does the document contain the exact words I typed?"*

    **Vector search:** *"Does the document mean something similar to what I typed?"*
    """)

    # Knowledge base
    documents = [
        "Black holes are regions of spacetime where gravity is so strong that nothing can escape.",
        "DNA carries the genetic instructions for the development and functioning of all living organisms.",
        "Machine learning algorithms learn patterns from data without being explicitly programmed.",
        "Quantum computers use qubits that can exist in multiple states simultaneously.",
        "Neural networks are computing systems inspired by the biological neurons in the human brain.",
        "Photosynthesis is the process by which plants convert sunlight into chemical energy.",
        "The theory of relativity describes the relationship between space, time, and gravity.",
        "Antibiotics are medicines that fight bacterial infections in humans and animals.",
        "Climate change refers to long-term shifts in global temperatures and weather patterns.",
        "The Internet connects billions of devices worldwide through a network of networks.",
    ]

    st.markdown("**Documents in our knowledge base:**")
    for i, doc in enumerate(documents, 1):
        st.markdown(f"{i}. {doc}")

    st.divider()

    query = st.text_input("Search query",
                          placeholder="e.g., 'How do computers learn?'")
    num_results = st.slider("Number of results", 1, 5, 3, key="num_results")

    if st.button("Search", type="primary") and query:
        model = load_model()

        with st.spinner("Building vector database and searching..."):
            from langchain_community.vectorstores import Chroma
            vectorstore = Chroma.from_texts(texts=documents, embedding=model)
            results = vectorstore.similarity_search_with_relevance_scores(
                query, k=num_results
            )

        st.subheader("Results (ranked by meaning similarity)")
        for i, (doc, score) in enumerate(results, 1):
            pct = max(0, score) * 100
            col_text, col_score = st.columns([4, 1])
            with col_text:
                st.markdown(f"**{i}.** {doc.page_content}")
            with col_score:
                st.metric("Relevance", f"{pct:.0f}%")
            st.progress(min(1.0, max(0, score)))

        st.divider()
        st.info("""
        **Try these queries to see semantic search in action:**
        - "How do computers learn?" -- finds ML and neural network docs
        - "What makes plants grow?" -- finds photosynthesis
        - "Space and the universe" -- finds black holes and relativity
        - "Fighting disease" -- finds antibiotics
        - "Global warming" -- finds climate change
        """)

### Run the app now!

Go to your browser and click **Rerun** to see the complete app with all pages working.

---

## Step 10: Deploy to Streamlit Cloud

Streamlit Cloud is a **free** hosting platform that deploys your app directly from GitHub.

### Deployment steps:

1. **Push your code to GitHub** (Step 8 above)
2. Go to **[share.streamlit.io](https://share.streamlit.io)**
3. Sign in with your **GitHub account**
4. Click **"New app"**
5. Select your repository (`ai-explorer-app`)
6. Set the main file path to `app.py`
7. Click **"Deploy"**
8. Wait a few minutes for the app to build
9. Share the URL with your classmates!

Your app will be live at a URL like: `https://your-username-ai-explorer-app.streamlit.app`

### Important notes about deployment:

- The `requirements.txt` file is **required** -- Streamlit Cloud uses it to install dependencies
- Every time you push to GitHub, the app **automatically updates**
- Free tier has resource limits -- the embedding model may be slow on free hosting
- For heavier workloads, consider deploying to **Render** or **Railway** instead

---

## Bonus: Connecting GitHub MCP in Gemini CLI

If you want to manage your GitHub repository using AI, you can connect the **GitHub MCP server** to Gemini CLI (or Claude CLI). This lets you create issues, make commits, and manage pull requests using natural language.

### Setup

1. Install Gemini CLI: `npm install -g @anthropic-ai/gemini-cli` (or use Claude Code)

2. Create a configuration file at `~/.gemini/settings.json`:

```json
{
  "mcpServers": {
    "github": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-github"],
      "env": {
        "GITHUB_PERSONAL_ACCESS_TOKEN": "ghp_your_token_here"
      }
    }
  }
}
```

3. Generate a GitHub Personal Access Token at **github.com/settings/tokens** with `repo` scope.

### Usage examples

Once connected, you can use natural language:
- *"Create an issue: Add dark mode toggle to the sidebar"*
- *"Show me the open issues on my ai-explorer-app repo"*
- *"Create a pull request for the new chunking feature"*

This is an example of **MCP (Model Context Protocol)** -- a standard for connecting AI tools to external services.

---

## Summary

In this notebook you built a complete Streamlit web application from scratch. Here is what you learned:

| Step | What you built | Key Streamlit concepts |
|------|---------------|----------------------|
| 1 | Hello World | `st.title()`, `st.write()`, `st.text_input()` |
| 2 | Multi-page navigation | `st.sidebar`, `st.set_page_config()`, `if/elif` routing |
| 3 | Embeddings comparison | `@st.cache_resource`, `st.button()`, `st.metric()`, `st.expander()` |
| 4 | Similarity matrix | `st.text_area()`, `st.plotly_chart()`, Plotly heatmaps |
| 5 | Chunking strategies | `st.tabs()`, `st.slider()`, widget keys |
| 6 | Vector search | ChromaDB, `st.spinner()`, `st.progress()` |
| 7 | Custom styling | CSS injection with `unsafe_allow_html=True` |
| 8 | GitHub integration | `git init`, `git push`, `requirements.txt` |
| 9 | Complete app | All pieces combined |
| 10 | Deployment | Streamlit Cloud from GitHub |

### Key takeaways:

- **Streamlit reruns the entire script** on every interaction -- this is its core execution model
- **Use `@st.cache_resource`** for expensive operations (loading models, database connections)
- **Every widget returns a value** -- `st.text_input()` returns the text, `st.slider()` returns the number, `st.button()` returns True/False
- **Plotly charts are interactive** -- much better than static matplotlib for web apps
- **ChromaDB + LangChain** make vector search surprisingly simple

### Next steps:

- Add a **file upload** feature (`st.file_uploader()`) so users can chunk their own documents
- Add a **text classification** page using embeddings + scikit-learn
- Connect to a **real LLM** (like Ollama) for a full RAG pipeline
- Try different embedding models and compare their quality

---

### Cleanup

When you are done, stop the Streamlit server:

In [ ]:
# Stop the Streamlit server
import sys, os
if sys.platform == "win32":
    os.system("taskkill /f /im streamlit.exe 2>nul")
else:
    os.system("pkill -f 'streamlit run' 2>/dev/null")
print("Streamlit server stopped.")